# Validation — `kappa-lora-metamathqa`

**What this measures:** Per the refusal fix (user_guidance), the only change is pointing `runner:` at the harness's Python entrypoint `method_comparison/MetaMathQA/run.py` instead of the non-Python Makefile; the target `num_trainable_params`, emitted by that harness for the PR's `experiments/kappa-lora/llama-3.2-3B-rank32` config, directly measures κ-LoRA's condition-number selection halving trainable parameters against the published 9,175,040 standard-LoRA row, with `test_accuracy` as the no-loss-of-fit guardrail — same suite kind, metrics, thresholds, baseline, and compute budget as before.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`14a295efe2f0`](https://github.com/mayorquinmachines/peft/commit/14a295efe2f031e74dbaf708a5c2bd16b93c9372)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [1]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "32cfaac8f9f3923496a2cf073a603e78e7bd0e94"
seed = 0


## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [3]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

GPU 0: NVIDIA L4 (UUID: GPU-5faeff98-9198-419f-4736-c52f8057e442)


python 3.12.3 · torch 2.14.0+cu126 · cuda True


## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [4]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "14a295efe2f031e74dbaf708a5c2bd16b93c9372"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

/workspace/target_repo
32cfaac Remyx: propose .remyx/validation.yaml for this change


## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [5]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

HF_TOKEN set


## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/kappa-lora/llama-3.2-3B-rank32` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/kappa-lora/llama-3.2-3B-rank32/adapter_config.json`:

```json
{"peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": "meta-llama/Llama-3.2-3B", "inference_mode": false, "r": 32, "target_modules": ["v_proj", "q_proj"], "condition_number_top_fraction": 0.5}
```

In [6]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/kappa-lora/llama-3.2-3B-rank32/adapter_config.json")).read())

{"peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": "meta-llama/Llama-3.2-3B", "inference_mode": false, "r": 32, "target_modules": ["v_proj", "q_proj"], "condition_number_top_fraction": 0.5}


## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [7]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

32cfaac8f9f3923496a2cf073a603e78e7bd0e94


## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — a directory of experiments runs each in turn; a single experiment runs once.

In [8]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/kappa-lora/llama-3.2-3B-rank32/*/")) or ["experiments/kappa-lora/llama-3.2-3B-rank32"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

[remyx] experiments/kappa-lora/llama-3.2-3B-rank32


/root/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:  50%|█████     | 1/2 [00:07<00:07,  7.55s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:20<00:00, 10.30s/it]

...

 93%|█████████▎| 25/27 [11:07<00:53, 26.80s/it]

 96%|█████████▋| 26/27 [11:38<00:28, 28.15s/it]

100%|██████████| 27/27 [11:47<00:00, 22.48s/it]

100%|██████████| 27/27 [11:47<00:00, 26.22s/it]

accelerator memory max: 20216MB
accelerator memory reserved avg: 13236MB
accelerator memory reserved 99th percentile: 18342MB
train time: 2688.730405818s
total time: 3268.39s
file size of checkpoint: 21.0MB


## 7. Read what the benchmark wrote

Results land under `results/*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [9]:
import glob, json, os
PATTERNS = ["results/*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

result document: temporary_results/kappa-lora--llama-3.2-3B-rank32--2026-09-11T23-28-13+00-00.json
{
  "num_trainable_params": 5505024,
  "test_accuracy": 0.39727065959059893
}


## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [10]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5000000,
        "baseline": 9175040
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.47052312357846854,
        "baseline": 0.49052312357846856
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    print(f"{c['metric']:<28}{str(v):>16}{str(c['baseline']):>16}  {c['direction']} {t}  {mark}")

metric                              observed        baseline  criterion
num_trainable_params                 5505024         9175040  <= 5000000  FAIL
test_accuracy               0.397270659590598930.49052312357846856  >= 0.47052312357846854  FAIL


## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [11]:
print(json.dumps(observed))

{"num_trainable_params": 5505024, "test_accuracy": 0.39727065959059893}


## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5000000 with `test_accuracy` >= 0.47052312357846854 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
benchmarks:
  - name: kappa-lora-metamathqa
    suite:
      harness:
        notebook: .remyx/validations/kappa-lora-metamathqa.ipynb
        runner: "method_comparison/MetaMathQA/run.py"
        experiments: "experiments/kappa-lora/llama-3.2-3B-rank32"
        results_glob: "results/*.json"
        method: "lora"
      scorer: "num_trainable_params"
      metrics:
        - name: "num_trainable_params"
          direction: min
          # derivation: standard LoRA row = 28 layers x [32*(3072+3072) for q + 32*(3072+1024) for v] = 9,175,040.
          # Balanced top-28-of-56 selection (14 q / 14 v) gives exactly 4,587,520; q modules carry 196,608 params
          # vs 131,072 for v, so 5,000,000 still admits up to a 20q/8v skew while staying within ~9% of exact half.
          threshold: 5000000
          role: target
        - name: "test_accuracy"
          direction: max
          # derivation: floor = the published row's own test_accuracy (0.49052312357846856 in
          # lora--llama-3.2-3B-rank32.json) minus a 2pp tolerance band, the repo's working definition of
          # "without losing fit"; it sits strictly inside the row's value, so the baseline row clears it by
          # construction (0.4905... > 0.4705...).
          threshold: 0.47052312357846856
          role: guardrail
    policy:
      guardrail_veto: true
    baseline:
      source: "method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      values:
        # deterministic count: 28 layers x (32x6144 q + 32x4096 v) = 28 x 327,680
        num_trainable_params: 9175040
        # read exactly from the corpus row lora--llama-3.2-3B-rank32.json; the published row is the authority
        test_accuracy: 0.49052312357846856
    held_constant:
      - "base model meta-llama/Llama-3.2-3B and the default_training_params.json protocol (seed, steps, batch size, max_seq_length, lr schedule) exactly as in the published lora--llama-3.2-3B-rank32 row"
      - "r=32 and the same q_proj/v_proj target-module set as the standard LoRA row; condition_number_top_fraction=0.5 is the only intended difference"
    avoid:
      - "changing r, target_modules, or any training param between the PR config and the compared LoRA row"
      - "substituting a synthetic CPU proxy for the MetaMathQA fit measurement"
      - "unpinned or swapped base-model revisions between runs"
    compute:
      tier: gpu
      # one arm = full MetaMathQA train+eval of Llama-3.2-3B with LoRA r=32 on one GPU, matching the runtimes
      # published in sibling corpus rows (~3-5 h); 6 h budget with margin
      timeout_s: 21600
    provenance:
      num_trainable_params: "user_guidance"
      test_accuracy: "user_guidance"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      held_constant: "protocol_doc:method_comparison/README.md"
      baseline: "published corpus:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      timeout_s: "published corpus runtimes"
```